# Number Detector Neural Network

The following code's purpose is to create a neural network that can detect what number is being written.

The steps to this neural network consists of 5 steps.

### 1. Set up the environment and the data.
We are using numpy and matplotlib as our libraries to help us create the neural netork. In addition, we need to download the database and preprocess it by flattening the data. The reason why we are flattening it is that we need it to be a 1D array so that the matrix math works out since we are multiplying the matrix by a weights matrix with 1 column. As a result, this is the only way for the math to work out.

### 2. Initialize W and b
W is our weight and b is our bias.

### 3. Define our math functions

## Environment setup and data import

In [123]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
import pandas as pd

mnist = fetch_openml('mnist_784')

## Initialize W and b

We initialize our weights, W1 and W2, where W1 represents different learning components of the shapes and W2 represnts the numbers 0-9. We initally first use 10 neurons for W1 since our network is pretty simple. Our weights are randomly assigned using normal distribution and multiplied by 0.01 so that they aren't too big and so that they represent different shapes. Our biases, b1 and b2, are set to 0 and adjusted during training.

In [124]:
def init_params(size):
    w1 = np.random.randn(size , 784) * 0.01
    b1 = np.zeros((size, 1))
    w2 = np.random.randn(size, size) * 0.01
    b2 = np.zeros((size, 1))

    return w1, b1, w2, b2

w1, b1, w2, b2 = init_params(0)

print(f"W1 shape: {w1.shape}")
print(f"W2 shape: {w2.shape}")

W1 shape: (0, 784)
W2 shape: (0, 0)


## Define our math functions

#### ReLU
ReLU is defined as max(0, Z). The reason for this is because we want to ensure that whatever Z output we get is positive. This solves the issue of negative Z's, since if we know that an output is bad and is negative, we don't want it affecting the other values that are created. This eliminates the noise of the negative numbers.

### Softmax
Softmax is an exponential function, so that means it will be louder if it is very wrong while quieter if it it's close. Additionally, it solves the issue of when the sum is 0 by making it e^Z.

### ReLU_deriv

In [125]:
def ReLU(Z):
    return np.maximum(0, Z)

def softmax(Z):
    exp = np.exp(Z - np.max(Z, axis=0, keepdims=True))
    return exp / np.sum(exp, axis = 0)

def ReLU_deriv(Z):
    return Z > 0


In [ ]:
def forward_prop(w1, b1, w2, b2, x):
    z1 = np.dot(w1, x) + b1
    a1 = ReLU(z1)

    z2 = np.dot(w2, a1) + b2
    a2 = softmax(z2)

    return z1, a1, z2, a2

def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, 10))
    one_hot_Y[np.arange(Y.size), Y] = 1
    return one_hot_Y.T

def backward_prop(z1, a1, z2, a2, w1, w2, X, Y, size):
    m = Y.size

    one_hot_Y = one_hot(Y, size)

    dZ2 = a2 - one_hot_Y 
    dW2 = 1 / m * dZ2.dot(a1.T)
    db2 = 1 / m * np.sum(dZ2, axis = 1, keepdims = True)

    dZ1 = w2.T.dot(dZ2) * ReLU_deriv(z1)

    dW1 = 1 / m * dZ1.dot(X.T)
    db1 = 1 / m * np.sum(dZ1, axis = 1, keepdims = True)
    
    return dW1, db1, dW2, db2

def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
    W1 = W1 - alpha * dW1
    b1 = b1 - alpha * db1    
    W2 = W2 - alpha * dW2  
    b2 = b2 - alpha * db2    
    return W1, b1, W2, b2



Preprocess

In [132]:
X_raw = mnist.data
Y_raw = mnist.target

if isinstance(Y_raw, pd.Series):
    Y = Y_raw.values
else:
    Y = np.array(Y_raw)

Y = Y.astype(np.int32).flatten()


if X_raw.shape[0] == 70000:
    X = X_raw.T
else:
    X = X_raw.T

X = np.array(X_raw).T / 255.0

m = X.shape[1]
n_train = 60000
X_train, X_test = X[:, :n_train], X[:, n_train:]
Y_train, Y_test = Y[:n_train], Y[n_train:]




In [130]:
epochs = 20
batch_size = 64
alpha = 0.1
size = 5

w1, b1, w2, b2 = init_params(size)

def cross_entropy_loss(a2, Y):
    m = Y.size
    one_hot_Y = one_hot(Y, size)

    return -np.sum(one_hot_Y * np.log(a2 + 1e-8)) / m


n_train = X_train.shape[1]

for epoch in range(epochs):
    # Mini-batch loop
    for start in range(0, n_train, batch_size):
        end = start + batch_size
        X_batch = X_train[:, start:end]
        Y_batch = Y_train[start:end]

        z1, a1, z2, a2 = forward_prop(w1, b1, w2, b2, X_batch)
        dW1, db1, dW2, db2 = backward_prop(z1, a1, z2, a2, w1, w2, X_batch, Y_batch, size)
        w1, b1, w2, b2 = update_params(w1, b1, w2, b2, dW1, db1, dW2, db2, alpha)

    # Optional: print average training loss once per epoch
    z1, a1, z2, a2 = forward_prop(w1, b1, w2, b2, X_train)
    loss = cross_entropy_loss(a2, Y_train)
    print(f"Epoch {epoch + 1}/{epochs}, loss: {loss:.4f}")

IndexError: index 5 is out of bounds for axis 1 with size 5

In [ ]:
def get_predictions(a2):
    return np.argmax(a2, axis = 0)

def get_accuracy(predictions, Y):
    return np.sum(predictions == Y) / len(predictions)

z1, a1, z2, a2 = forward_prop(w1, b1, w2, b2, X_test)
predictions = get_predictions(a2)
accuracy = np.mean(predictions == Y_test)
print(f"Test accuracy: {accuracy * 100:.2f}%")

Test accuracy: 93.71%
